# Encoding TEI with LLMs

This notebook demonstrates how to use Large Language Models (LLMs) to assist in the encoding of texts in the Text Encoding Initiative (TEI) format. The TEI is a standard for representing texts in digital form, and LLMs can help automate and enhance the encoding process.


In [ ]:
from pathlib import Path
import os
from openai import OpenAI
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

In [ ]:
SYSTEM_PROMPT = """
Tu es un expert en encodage XML TEI pour les dictionnaires anciens. Ton rôle est de transformer du texte OCR brut du *Dictionnaire de Trévoux* (1743) en un fichier XML structuré et valide.

### RÈGLES ABSOLUES
- **Sortie exclusive :** Ne génère QUE le code XML valide. AUCUN commentaire, aucune explication avant ou après.
- **Fidélité stricte :** Ne rien inventer, corriger, moderniser ou compléter. Conserver la casse, la ponctuation et les fautes de l'OCR. Si c'est ambigu, encode tel quel.
- **Continuité :** Les entrées doivent suivre l'ordre exact du texte OCR. 
- **Fermeture :** Vérifie que chaque balise ouverte est fermée.

### STRUCTURE XML ATTENDUE
<TEI>
  <text>
    <body>
      <entry type="fragment" subtype="continued">
        <sense>
          <def>... suite du texte sans lemme ni form ...</def>
        </sense>
      </entry>
      ...
      <entry>
        <form type="lemma"><orth>LEMME</orth></form>
        <gramGrp><pos>s. m.</pos></gramGrp> <sense>
          <def>Texte de définition...</def>
          <cit type="example"><quote>Citation éventuelle</quote></cit>
        </sense>
      </entry>
    </body>
  </text>
</TEI>

### LOGIQUE D'ANALYSE ET D'ENCODAGE

**1. Gestion des entrées (`<entry>`)**
- **Nouvelle entrée :** Détectée par un mot/groupe de mots ENTIÈREMENT EN CAPITALES, situé en DÉBUT de paragraphe, suivi d'une ponctuation (, ou .) ou d'une marque grammaticale. (Attention : un mot en majuscules au milieu d'une phrase n'est pas un lemme).
- **Fin de page :** Ne jamais marquer une entrée comme incomplète à la fin du texte fourni. Considère-la toujours comme complète.
- **Cas Fragmentaire (Début de texte) :** Si le tout premier bloc du texte OCR ne commence pas par un lemme valide (ex: minuscule, phrase directe), c'est une continuation. Utilise `<entry type="fragment" subtype="continued">`. N'ajoute AUCUNE balise `<form>`.

**2. Lemme et Grammaire (`<orth>` et `<pos>`)**
- `<orth>` : Le premier segment en capitales.
- `<pos>` : S'il y a une abréviation grammaticale (ex: "s. m.", "v. act.") juste après le lemme, l'isoler dans `<pos>`. Sinon, omettre `<gramGrp>`.

**3. Sens et Définitions (`<sense>` et `<def>`)**
- Crée un nouveau `<sense>` UNIQUEMENT s'il y a un marqueur explicite (1., 2., I., II., A., B.) ou une rupture sémantique très claire. Sinon, un seul `<sense>` global.
- `<def>` : Contient le texte explicatif. Les renvois internes ("Voyez...", "V.") restent dans `<def>` sans balise spéciale.

**4. Citations (`<cit type="example">` > `<quote>`)**
- Encode en citation : les phrases en italique signalées par l'OCR, les citations latines, celles entre guillemets, ou introduites par un auteur. 
- Ne sépare jamais une même citation en plusieurs blocs.

### EXEMPLE DE RÉFÉRENCE
**OCR :** ABANDON. s. m. Action de délaisser. Ex. L'abandon d'un bien.
**XML :**
<entry>
  <form type="lemma"><orth>ABANDON</orth></form>
  <gramGrp><pos>s. m.</pos></gramGrp>
  <sense>
    <def>Action de délaisser. Ex.</def>
    <cit type="example"><quote>L'abandon d'un bien.</quote></cit>
  </sense>
</entry>
"""


def build_user_prompt(text: str) -> str:
    return f"""TEXTE À ENCODER :
{text}

TEXTE ENCODE EN XML TEI LEX-0 :
"""


def openrouter(client, model_name, text):
    user_prompt = build_user_prompt(text)
    # Gemma → pas de system message
    if "gemma" in model_name.lower() or "mistral" in model_name.lower():

        full_prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "user", "content": full_prompt},
            ],
        )

    # Autres modèles → messages normaux
    else:
        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )

    return response.choices[0].message.content.strip()


def tei_encoding(text, provider, client, model_name):
    try:
        if provider == "openrouter":
            return openrouter(client, model_name, text)
        else:
            raise ValueError("Provider inconnu")

    except Exception as e:
        print(f"Erreur {provider}: {e}")
        return text

True

In [ ]:
'''
openrouter_client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm_studio",
)
models = ["mistralai/mistral-7b-instruct-v0.3"]
'''

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
models = ["openai/gpt-5-mini"] #, "google/gemma-3-27b-it", "google/gemma-3-12b-it"]

#ocr_path_name = "data/glm-ocr/txt/"
ocr_path_name = "data/robert/pages-txt/"
ocr_files = sorted(Path(ocr_path_name).glob("*.txt"))

for model_name in models:
    model_short = model_name.split("/")[1].split(":")[0]

    # create folder with model name if it doesn't exist
    output_path = Path(f"data/encoded/{model_short}")
    output_path.mkdir(parents=True, exist_ok=True)

    for ocr_file in tqdm(ocr_files):
        ocr_text = ocr_file.read_text(encoding="utf-8", errors="replace")

        # if file already exists, skip
        output_file = output_path / f"{ocr_file.stem}.xml"
        if output_file.exists():
            continue
        corrected_text = tei_encoding(ocr_text, "openrouter", openrouter_client, model_name)

        # Write corrected text to output file
        output_file.write_text(corrected_text, encoding="utf-8", errors="replace")



100%|██████████| 2/2 [02:34<00:00, 77.46s/it]


In [ ]:
# =====================================================
# USAGE (example de test rapide)
# =====================================================

text = "Ceci est un eximp1e de texte OQR avec des emeurs."

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

model_name = "google/gemma-3-27b-it"

print("OpenRouter (gemma):", tei_encoding(text, "openrouter", openrouter_client, model_name))